# 01 — Read-only аудит `MGTS_data`

Цель notebook — воспроизводимо описать структуру и качество `MGTS_data/*.csv` до выбора ML-постановки. Он **только читает** исходные CSV: не исправляет значения, не создаёт производные датасеты, не обучает модели и не применяет FFT/STFT.

Правила интерпретации:

- названия физических осей и единицы пока неизвестны, поэтому используются нейтральные `x` и `y`;
- `-300` рассматривается как технический маркер, а не как физическая температура;
- временной разрыв — диагностический интервал `> 1.5 × median(Δt)`, а не подтверждённая потеря данных;
- статистический выброс — кандидат на проверку, а не автоматически аномальное событие.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import random
from collections import Counter, defaultdict
from datetime import datetime
from itertools import pairwise
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from mgts_sensing.io import iter_profiles

SEED = 42
SENTINEL = -300.0
csv.field_size_limit(2**31 - 1)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / "MGTS_data").is_dir():
            return candidate
    raise FileNotFoundError("Не найдена папка MGTS_data; откройте notebook из репозитория")

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "MGTS_data"
CSV_FILES = sorted(RAW_DIR.glob("*.csv"), key=lambda path: int(path.stem))
assert CSV_FILES, f"CSV не найдены в {RAW_DIR}"

# Снимок размера и mtime нужен только для финальной проверки read-only режима.
raw_state_before = {p.name: (p.stat().st_size, p.stat().st_mtime_ns) for p in CSV_FILES}
print(f"Проект: {PROJECT_ROOT}")
print(f"Найдено CSV: {len(CSV_FILES)}")

## 1. Схема и смысл одной строки

Одна строка CSV описывает **один пространственный профиль одного канала в один момент времени**:

- `dts_id` — идентификатор источника/устройства; точный физический смысл требуется подтвердить;
- `channel_id` — номер канала;
- `timestamp` — время измерения; форматы timezone различаются;
- `points` — JSON-массив пар `[x, y]`, где `x` — координата вдоль профиля, `y` — измеренное значение; единицы неизвестны;
- `points_size` — заявленная длина массива, присутствует не во всех файлах.

Ниже показывается только безопасное краткое представление строки — огромный массив `points` целиком не печатается.

In [ ]:
with CSV_FILES[0].open(newline="", encoding="utf-8") as handle:
    example_row = next(csv.DictReader(handle))
example_points = json.loads(example_row["points"])

example = {
    "dts_id": example_row["dts_id"],
    "channel_id": example_row["channel_id"],
    "timestamp": example_row["timestamp"],
    "points_size_declared": example_row.get("points_size"),
    "points_size_actual": len(example_points),
    "first_3_points": example_points[:3],
    "last_3_points": example_points[-3:],
}
display(pd.Series(example, name="пример строки").to_frame())

## 2. Потоковый аудит

CSV читаются по одной строке, поэтому все профили не накапливаются в памяти. Для графиков сохраняются только первый, два соседних и четыре детерминированно выбранных профиля каждого файла. Offset вида `+03` временно дополняется до `+03:00` только при разборе времени.

In [ ]:
def compact_counter(counter: Counter, limit: int = 6) -> str:
    items = sorted(counter.items())
    if len(items) <= limit:
        return ", ".join(f"{key}×{count}" for key, count in items)
    mode, mode_count = counter.most_common(1)[0]
    return f"{min(counter)}…{max(counter)}; mode={mode}×{mode_count}; variants={len(counter)}"

def audit_csv(path: Path) -> tuple[dict, dict, dict]:
    rng = random.Random(SEED + int(path.stem))
    ids, channels = Counter(), Counter()
    lengths, declared_lengths = Counter(), Counter()
    dx_counts, grids = Counter(), Counter()
    timestamps_by_stream: dict[tuple[str, str], list[datetime]] = defaultdict(list)
    profile_means: list[float] = []
    first_two: list[dict] = []
    reservoir: list[dict] = []
    first_profile = None

    counters = Counter()
    x_min, x_max = math.inf, -math.inf
    y_min, y_max = math.inf, -math.inf
    valid_y_min, valid_y_max = math.inf, -math.inf
    fields: list[str] = []

    # Display schema, but delegate all profile parsing to the shared ingestion module.
    with path.open(newline="", encoding="utf-8-sig") as handle:
        fields = next(csv.reader(handle))
    issue_codes = {
        "MISSING_REQUIRED_FIELD": "missing_required", "DUPLICATE_KEY": "duplicate_keys",
        "INVALID_TIMESTAMP": "bad_timestamps", "INVALID_JSON": "bad_json",
        "INVALID_POINTS_STRUCTURE": "bad_json", "INVALID_POINT_PAIR": "bad_point_shape",
        "NONNUMERIC_POINT": "nonnumeric_points", "NONFINITE_POINT": "nonfinite_points",
        "NONINCREASING_X": "nonincreasing_x", "POINTS_SIZE_MISMATCH": "size_mismatch",
        "INVALID_POINTS_SIZE": "size_mismatch",
    }
    for result in iter_profiles([path]):
        if result.row_number > 0:
            counters["rows"] += 1
        for issue in result.issues:
            if issue.code in issue_codes:
                counters[issue_codes[issue.code]] += 1
        if not result.loaded:
            print(f"Rejected {path.name}:{result.line_number}: "
                  + ", ".join(issue.code for issue in result.issues))
            counters["rejected"] += int(result.row_number > 0)
            continue
        profile = result.profile
        dts_id, channel = profile.dts_id, profile.channel_id
        timestamp, raw_ts = profile.timestamp, profile.timestamp_raw
        row_index = result.row_number
        ids[dts_id] += 1
        channels[channel] += 1
        timestamps_by_stream[(dts_id, channel)].append(timestamp)
        counters["short_offset_rows"] += int(profile.timestamp_format_normalized)
        x_array, y_array = profile.x, profile.y
        lengths[len(x_array)] += 1
        if profile.points_size is not None:
            declared_lengths[profile.points_size] += 1
        diffs = np.diff(x_array)
        dx_counts.update(np.round(diffs[diffs > 0], 9).tolist())
        grids[hashlib.sha1(x_array.tobytes()).hexdigest()] += 1
        if len(x_array):
            x_min, x_max = min(x_min, float(x_array.min())), max(x_max, float(x_array.max()))
        if len(y_array):
            y_min, y_max = min(y_min, float(y_array.min())), max(y_max, float(y_array.max()))
            sentinel_mask = y_array == SENTINEL
            counters["sentinel_points"] += int(sentinel_mask.sum())
            counters["sentinel_rows"] += int(sentinel_mask.any())
            valid_values = y_array[~sentinel_mask]
            if len(valid_values):
                valid_y_min = min(valid_y_min, float(valid_values.min()))
                valid_y_max = max(valid_y_max, float(valid_values.max()))
                profile_means.append(float(valid_values.mean()))
            trailing = 0
            for value in y_array[::-1]:
                if value != SENTINEL:
                    break
                trailing += 1
            counters[f"trailing_sentinel_{trailing}"] += 1

        sample = {"timestamp": raw_ts, "parsed_timestamp": timestamp, "x": x_array, "y": y_array}
        if first_profile is None:
            first_profile = sample
        if len(first_two) < 2:
            first_two.append(sample)
        if len(reservoir) < 4:
            reservoir.append(sample)
        else:
            replacement = rng.randrange(row_index)
            if replacement < 4:
                reservoir[replacement] = sample

    all_times, intervals = [], []
    ordered = True
    duplicate_timestamps = 0
    for stream_times in timestamps_by_stream.values():
        all_times.extend(stream_times)
        ordered &= stream_times == sorted(stream_times)
        duplicate_timestamps += len(stream_times) - len(set(stream_times))
        sorted_times = sorted(stream_times)
        intervals.extend((b - a).total_seconds() for a, b in pairwise(sorted_times))

    median_interval = float(np.median(intervals)) if intervals else math.nan
    gaps = [value for value in intervals if value > 1.5 * median_interval]
    modal_length, _ = lengths.most_common(1)[0]
    missing_vs_modal = sum(max(0, modal_length - length) * count for length, count in lengths.items())

    means = np.asarray(profile_means)
    mean_median = float(np.median(means))
    mean_mad = float(np.median(np.abs(means - mean_median)))
    mean_outliers = int(np.sum(np.abs(means - mean_median) > 6 * mean_mad)) if mean_mad else 0

    tz_labels = Counter(
        "naive" if stamp.tzinfo is None else str(stamp.utcoffset()) for stamp in all_times
    )
    summary = {
        "file": path.name,
        "size_MiB": round(path.stat().st_size / 2**20, 1),
        "schema": ", ".join(fields),
        "profiles": counters["rows"],
        "dts_id": ", ".join(ids),
        "channels": ", ".join(sorted(channels)),
        "time_start": min(all_times).isoformat(timespec="seconds"),
        "time_end": max(all_times).isoformat(timespec="seconds"),
        "timezone": ", ".join(tz_labels),
        "dt_min_s": min(intervals),
        "dt_median_s": median_interval,
        "dt_max_s": max(intervals),
        "diagnostic_gaps": len(gaps),
        "points_lengths": compact_counter(lengths),
        "points_mode": modal_length,
        "short_profiles": sum(count for length, count in lengths.items() if length < modal_length),
        "missing_tail_points_vs_mode": missing_vs_modal,
        "x_min": x_min,
        "x_max": x_max,
        "x_step_mode": dx_counts.most_common(1)[0][0],
        "grid_variants": len(grids),
        "y_min": y_min,
        "y_max": y_max,
        "valid_y_min": valid_y_min,
        "valid_y_max": valid_y_max,
        "sentinel_points": counters["sentinel_points"],
        "profile_mean_outliers_6MAD": mean_outliers,
    }
    quality = {
        "file": path.name,
        "rejected": counters["rejected"],
        "missing_required": counters["missing_required"],
        "duplicate_keys": counters["duplicate_keys"],
        "duplicate_timestamps": duplicate_timestamps,
        "bad_timestamps": counters["bad_timestamps"],
        "short_offset_rows": counters["short_offset_rows"],
        "bad_JSON": counters["bad_json"],
        "bad_point_shape": counters["bad_point_shape"],
        "nonnumeric_points": counters["nonnumeric_points"],
        "nonfinite_points": counters["nonfinite_points"],
        "nonincreasing_x": counters["nonincreasing_x"],
        "points_size_mismatch": counters["size_mismatch"],
        "ordered_by_time": ordered,
    }
    details = {
        "lengths": lengths,
        "declared_lengths": declared_lengths,
        "intervals": intervals,
        "gaps": gaps,
        "first_profile": first_profile,
        "first_two": first_two,
        "reservoir": sorted(reservoir, key=lambda item: item["parsed_timestamp"]),
        "counters": counters,
        "profile_means": means,
        "mean_median": mean_median,
        "mean_mad": mean_mad,
    }
    return summary, quality, details

In [ ]:
summaries, quality_rows, audit_details = [], [], {}
for csv_path in CSV_FILES:
    print(f"Аудит {csv_path.name} ...")
    summary, quality, details = audit_csv(csv_path)
    summaries.append(summary)
    quality_rows.append(quality)
    audit_details[csv_path.name] = details

summary_df = pd.DataFrame(summaries).set_index("file")
quality_df = pd.DataFrame(quality_rows).set_index("file")
print("Готово. Исходные CSV не изменялись.")

## 3. Паспорт файлов

In [ ]:
display(summary_df[[
    "size_MiB", "profiles", "dts_id", "channels",
    "time_start", "time_end", "timezone",
    "dt_min_s", "dt_median_s", "dt_max_s", "diagnostic_gaps",
]])

In [ ]:
display(summary_df[[
    "points_lengths", "short_profiles", "missing_tail_points_vs_mode",
    "x_min", "x_max", "x_step_mode", "grid_variants",
    "valid_y_min", "valid_y_max", "sentinel_points",
]])

## 4. Пропуски, дубликаты и невалидные значения

`missing_tail_points_vs_mode` — сколько точек не хватает в укороченных профилях относительно модальной длины файла. Это не равно подтверждённым пропускам сенсора: короткая запись может быть допустимым форматом.

In [ ]:
display(quality_df)

length_rows = []
for file_name, details in audit_details.items():
    for length, count in sorted(details["lengths"].items()):
        length_rows.append({"file": file_name, "points_length": length, "profiles": count})
display(pd.DataFrame(length_rows).set_index(["file", "points_length"]))

### Статистические кандидаты в выбросы

Для каждого профиля вычисляется среднее по значениям, отличным от `-300`. Кандидатом считается профиль дальше `6 × MAD` от медианы средних. Такой флаг полезен для ручного просмотра, но **не является физической разметкой аномалии** и не должен превращаться в target без эксперта.

In [ ]:
outlier_table = summary_df[["profile_mean_outliers_6MAD", "valid_y_min", "valid_y_max"]].copy()
outlier_table["profile_mean_median"] = [audit_details[name]["mean_median"] for name in outlier_table.index]
outlier_table["profile_mean_MAD"] = [audit_details[name]["mean_mad"] for name in outlier_table.index]
display(outlier_table)

## 5. Технический маркер `-300` в `5.csv`

Маркер визуализируется отдельно от сигнала. На верхнем графике позиции с `-300` заменены на `NaN`; нижний график показывает только бинарную маску технического маркера.

In [ ]:
profile_5 = audit_details["5.csv"]["first_profile"]
x5, y5 = profile_5["x"], profile_5["y"]
marker_mask = y5 == SENTINEL
visible_y = y5.copy()
visible_y[marker_mask] = np.nan

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, height_ratios=[3, 1])
axes[0].plot(x5, visible_y, linewidth=1)
axes[0].set(title=f"5.csv, {profile_5['timestamp']}: значения без технического маркера", ylabel="y, единицы неизвестны")
axes[1].fill_between(x5, marker_mask.astype(int), step="mid", color="tab:red", alpha=0.7)
axes[1].set(xlabel="x, единицы неизвестны", ylabel="y == -300", yticks=[0, 1])
fig.suptitle("-300 показан как техническая маска, а не как температура")
fig.tight_layout()
plt.show()

print(f"Маркерных точек: {int(marker_mask.sum())} из {len(marker_mask)} ({marker_mask.mean():.2%})")

## 6. Один профиль `y(x)` из каждого файла

Для `5.csv` технический хвост `-300` исключён из линии. Графики нельзя сравнивать как температуру до подтверждения единиц и типа сенсоров.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, file_name in zip(axes.flat, summary_df.index):
    profile = audit_details[file_name]["first_profile"]
    mask = profile["y"] != SENTINEL
    ax.plot(profile["x"][mask], profile["y"][mask], linewidth=0.9)
    ax.set(title=f"{file_name} · {profile['timestamp']}", xlabel="x", ylabel="y")
fig.suptitle("Первые профили файлов; физические единицы осей неизвестны")
fig.tight_layout()
plt.show()

## 7. Несколько профилей во времени

Для примера выбран `3.csv`: у него одна пространственная сетка и нет `-300`. Четыре профиля выбраны детерминированным reservoir sampling с seed `42`.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for profile in audit_details["3.csv"]["reservoir"]:
    ax.plot(profile["x"], profile["y"], linewidth=0.9, label=profile["timestamp"])
ax.set(title="3.csv: профили в разные моменты времени", xlabel="x, единицы неизвестны", ylabel="y, единицы неизвестны")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 8. Разность соседних профилей `Δy(x)`

Показана разность второго и первого последовательных профилей `3.csv`. Одинаковая сетка проверяется до вычитания. Знак: `y(t₂, x) − y(t₁, x)`.

In [ ]:
previous_profile, current_profile = audit_details["3.csv"]["first_two"]
assert np.array_equal(previous_profile["x"], current_profile["x"]), "Сетки соседних профилей различаются"
delta_y = current_profile["y"] - previous_profile["y"]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(current_profile["x"], delta_y, linewidth=0.9)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(
    title=f"Δy(x): {current_profile['timestamp']} − {previous_profile['timestamp']}",
    xlabel="x, единицы неизвестны",
    ylabel="Δy, единицы неизвестны",
)
fig.tight_layout()
plt.show()

## 9. Выводы

### Подтверждено данными

- Файлы содержат временные последовательности пространственных профилей, но имеют разные сетки, длины и частоты измерений.
- `points_size` присутствует только в `1.csv` и `6.csv`; там он совпадает с фактической длиной.
- Обязательные поля заполнены; JSON читается; нечисловые точки, `NaN`, `Inf`, нарушения порядка `x` и дубликаты ключей не обнаружены.
- В `2.csv`, `4.csv` и `6.csv` есть укороченные профили; недостаёт именно хвоста полной пространственной сетки.
- Временные интервалы нерегулярны, включая длинные разрывы.
- В каждой строке `5.csv` последние 675 значений равны `-300`; это 28.55% профиля.
- `5.csv` отличается форматом `dts_id`, отсутствием timezone, шагом `x = 1` и годом данных.

### Гипотезы, требующие проверки

- `-300` может быть padding/кодом недоступного участка, но это не доказано.
- Укороченные профили могут быть результатом неполной передачи или допустимой переменной длиной.
- Значения похожи по диапазону на температуру, но называть их температурой без словаря данных нельзя.
- Разрывы времени могут быть пропусками, режимом опроса или остановками оборудования.

### Риски leakage

- Нельзя случайно делить соседние строки или перекрывающиеся окна между train/test.
- Нормализация, импутация, пороги выбросов и подбор параметров должны обучаться только на train.
- `dts_id`, имя файла, канал, длина/шаг сетки, timestamp и наличие `-300` могут раскрывать источник файла вместо целевого события.
- Интерполяция не должна использовать будущие значения или test-интервалы.
- Файлы `2–4.csv` имеют общий `dts_id` и не должны автоматически считаться независимыми объектами.

### Вопросы организаторам `[HUMAN DECISION]`

1. Что физически означают `x`, `y`, `dts_id` и `channel_id`, каковы единицы?
2. Какие файлы относятся к DTS и DAS?
3. Что означает `-300` и нужно ли отбрасывать соответствующие позиции?
4. Какая timezone у `5.csv`, можно ли сопоставлять её с UTC/`+03`?
5. Какова штатная частота опроса и допустимы ли переменная длина и временные разрывы?
6. Есть ли разметка событий, интервалы аварий, обслуживания или внешний target?
7. Что является независимым объектом для validation split: устройство, линия, канал, дата или участок?

До ответов не выполняются обучение модели и FFT/STFT.

In [ ]:
raw_state_after = {p.name: (p.stat().st_size, p.stat().st_mtime_ns) for p in CSV_FILES}
assert raw_state_after == raw_state_before, "Состояние исходных CSV изменилось"
print("Read-only проверка пройдена: размеры и mtime всех исходных CSV не изменились.")